# Insurance 02 — 10,000 and 100,000 rows: analytical retrieval
## Question
Can the system compute an exhaustive total with a traceable scope, instead of summing an arbitrary top-k subset?

Generate two deterministic **synthetic** portfolios, import every row into SQLite and compare SQL outputs against independently accumulated generator totals. This notebook tests JSONL inputs. Continue in notebook 04 for the actual XLSX round-trip using the approved openpyxl dependency.

No API key, paid model or external database is required. Amounts are stored as integer EUR cents. Distributions are deliberately artificial, including outliers; these are not actuarial data.


In [1]:
from pathlib import Path
import sys, json, hashlib
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'backend/app').is_dir())
if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))
from IPython.display import display, Markdown, Image
import pandas as pd
BASE = ROOT / 'data/insurance_v2'
print('Isolated insurance workspace:', BASE)



Isolated insurance workspace: C:\Users\choun\Downloads\Prudential_Evidence_Lab_MVP_Source\prudential_evidence_lab\data\insurance_v2


## 1. Generate and fingerprint the inputs
There are five claims per policy and two synthetic tenants. Re-running reuses the same verified dataset; a changed file is rejected rather than overwritten.


In [2]:
from ingestion.insurance_scale import generate, benchmark, TENANTS, aggregate, import_rows
sizes = [10_000, 100_000]
for size in sizes:
    manifest = generate(BASE / f'synthetic/portfolios/claims_{size}', size)
    display({k: manifest[k] for k in ['generator','synthetic','rows','sha256','monetary_unit']})


{'generator': 'synthetic-claims-v1',
 'synthetic': True,
 'rows': 10000,
 'sha256': '76c99130d83b183ec9582fd87c9dda93859a2a34146563e57e92da1fed6519ae',
 'monetary_unit': 'integer cents'}

{'generator': 'synthetic-claims-v1',
 'synthetic': True,
 'rows': 100000,
 'sha256': 'bcf2bbc3d61defe1b41f631c3c9b5b131884c04105171d3b478e5eb04eb5b8b8',
 'monetary_unit': 'integer cents'}

## 2. Inspect actual records and field meaning
claim_id is the unique row key; policy_id links claims belonging to a policy; tenant_id is the mandatory query scope. claimed_cents and paid_cents have different meanings. Zero paid does not mean a rejected claim. The generated status is not an insurance decision. Files contain every row; the notebook shows a readable sample instead of embedding 110,000 rows.


In [3]:
source = BASE / 'synthetic/portfolios/claims_10000/claims.jsonl'
with source.open(encoding='utf-8') as stream:
    sample = [json.loads(next(stream)) for _ in range(12)]
display(pd.DataFrame(sample))
print('Full input:', source)


,claim_id,policy_id,tenant_id,loss_date,peril,status,claimed_cents,paid_cents,currency
0,SYN-C0000001,SYN-P0000001,SYNTHETIC_A,2025-01-01,water,open,15025000,0,EUR
1,SYN-C0000002,SYN-P0000001,SYNTHETIC_A,2025-02-02,theft,closed,32919,23043,EUR
2,SYN-C0000003,SYN-P0000001,SYNTHETIC_A,2025-03-03,fire,closed,40838,28586,EUR
3,SYN-C0000004,SYN-P0000001,SYNTHETIC_A,2025-04-04,storm,open,48757,0,EUR
4,SYN-C0000005,SYN-P0000001,SYNTHETIC_A,2025-05-05,water,closed,56676,39673,EUR
5,SYN-C0000006,SYN-P0000002,SYNTHETIC_B,2025-06-06,theft,closed,64595,45216,EUR
6,SYN-C0000007,SYN-P0000002,SYNTHETIC_B,2025-07-07,fire,open,72514,0,EUR
7,SYN-C0000008,SYN-P0000002,SYNTHETIC_B,2025-08-08,storm,closed,80433,56303,EUR
8,SYN-C0000009,SYN-P0000002,SYNTHETIC_B,2025-09-09,water,closed,88352,61846,EUR
9,SYN-C0000010,SYN-P0000002,SYNTHETIC_B,2025-10-10,theft,open,96271,0,EUR


Full input: C:\Users\choun\Downloads\Prudential_Evidence_Lab_MVP_Source\prudential_evidence_lab\data\insurance_v2\synthetic\portfolios\claims_10000\claims.jsonl


## 3. Measure import and query separately
Each run uses a fresh in-memory SQLite database. Timings depend on this machine and do not measure concurrency, Excel parsing, vector search or end-to-end answer latency. SQL runs in query-only mode after ingestion. Compare integer sums exactly, not with a floating tolerance.


In [4]:
runs = [benchmark(BASE / f'synthetic/portfolios/claims_{size}') for size in sizes]
display(pd.DataFrame([{k: r[k] for k in ['rows','import_seconds','query_seconds','sql_matches_gold']}
                      for r in runs]))
assert all(r['sql_matches_gold'] for r in runs)
display(pd.DataFrame([{'rows': r['rows'], 'tenant': tenant, **totals}
                      for r in runs for tenant, totals in r['by_tenant'].items()]))


,rows,import_seconds,query_seconds,sql_matches_gold
0,10000,0.210433,0.009590,True
1,100000,2.218210,0.123636,True


,rows,tenant,count,claimed_cents,paid_cents
0,10000,SYNTHETIC_A,5000,5888215000,2724469159
1,10000,SYNTHETIC_B,5000,5862190000,2754796023
2,100000,SYNTHETIC_A,50000,58982150000,27504961159
3,100000,SYNTHETIC_B,50000,58966900000,27536279023


## 4. Why ten selected rows cannot establish an exhaustive total
This is an illustrative first-ten-row subset, **not vector retrieval and not a measured RAG baseline**. Even a semantically excellent top-k retriever cannot assert it returned every accounting record. A total or mean needs a complete, scoped population; finding a policy clause needs documentary retrieval.


In [5]:
comparison = []
for run in runs:
    whole = run['by_tenant'][TENANTS[0]]
    part = run['illustrative_ten_row_subset']
    comparison.append({'rows_in_dataset': run['rows'], 'scope_rows': whole['count'],
                       'subset_rows': part['count'], 'exhaustive_total_eur': whole['claimed_cents']/100,
                       'subset_total_eur': part['claimed_cents']/100,
                       'exhaustive_mean_eur': whole['claimed_cents']/whole['count']/100,
                       'subset_mean_eur': part['claimed_cents']/part['count']/100})
display(pd.DataFrame(comparison))


,rows_in_dataset,scope_rows,subset_rows,exhaustive_total_eur,subset_total_eur,exhaustive_mean_eur,subset_mean_eur
0,10000,5000,10,58882150.0,158043.3,11776.43,15804.33
1,100000,50000,10,589821500.0,158043.3,11796.43,15804.33


## 5. Boundary tests: invalid rows and missing scope
Unit tests also reject duplicate keys, invalid dates, mixed currency and negative or non-integer amounts. Batch import is transactional: one invalid row rejects the whole incoming batch. Scope filtering is tested; identity/authentication and enterprise row-level security are not implemented here.


In [6]:
import sqlite3
from ingestion.insurance_scale import records
with sqlite3.connect(':memory:') as connection:
    import_rows(connection, records(20), source_name='synthetic fixture')
    assert aggregate(connection, tenant=TENANTS[0])['count'] == 10
    try:
        aggregate(connection, tenant="SYNTHETIC_A' OR 1=1 --")
    except ValueError as error:
        print('Rejected unknown scope:', error)
    else:
        raise AssertionError('Scope bypass')
print('Run pytest backend/tests/test_insurance_scale.py for the full boundary matrix.')


Rejected unknown scope: Explicit known tenant required; this is not authentication.
Run pytest backend/tests/test_insurance_scale.py for the full boundary matrix.


## Conclusion
Passing means exhaustive scoped SQL matches the generated truth at these two row counts. It does not mean general Excel ingestion, LLM-generated SQL, hybrid RAG answers or production capacity have been validated. Next: XLSX round-trip, formula handling, multiple sheets, explicit data-quality quarantine, then bounded analytical tools integrated with the evidence contract.
